In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import os
from glob import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from tqdm import tqdm

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Step 1: Set up device and parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

config = {
    "folder1_path": r"D:\Biose\After_Rework\1000_Dataset\image",
    "folder2_path": r"D:\Biose\After_Rework\1000_Dataset\label",
    "batch_size": 32,
    "learning_rate": 0.0001,
    "num_epochs": 50,
    "image_size": 224,
    "num_classes": 2,
    "k_folds": 5,
    "patience": 7  # Early stopping patience
}

# --- Early Stopping Utility ---
class EarlyStopping:
    def __init__(self, patience=7, delta=0):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_loss, model, path):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model, path)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(model, path)
            self.counter = 0

    def save_checkpoint(self, model, path):
        torch.save(model.state_dict(), path)

# --- Dataset Class ---
class PairedImageDataset(Dataset):
    def __init__(self, pairs, labels, transform=None):
        self.pairs = pairs
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path, img2_path = self.pairs[idx]
        label = self.labels[idx]
        img1 = Image.open(img1_path).convert('RGB')
        img2 = Image.open(img2_path).convert('RGB')
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return img1, img2, torch.tensor(label, dtype=torch.long)

# --- Model Architecture ---
class PairedResNet(nn.Module):
    def __init__(self, num_classes):
        super(PairedResNet, self).__init__()
        self.resnet = models.resnet18(weights='IMAGENET1K_V1')
        self.feature_extractor = nn.Sequential(*list(self.resnet.children())[:-1])
        
        # Freeze first 3 layers as per your requirement
        child_list = list(self.feature_extractor.children())
        for param in nn.Sequential(*child_list[:3]).parameters():
            param.requires_grad = False
            
        self.fc = nn.Sequential(
            nn.Linear(512 * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x1, x2):
        feat1 = torch.flatten(self.feature_extractor(x1), 1)
        feat2 = torch.flatten(self.feature_extractor(x2), 1)
        combined = torch.cat((feat1, feat2), dim=1)
        return self.fc(combined)

# --- Evaluation Function ---
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, tp, fp, tn, fn = 0, 0, 0, 0, 0
    with torch.no_grad():
        for i1, i2, labels in loader:
            i1, i2, labels = i1.to(device), i2.to(device), labels.to(device)
            outputs = model(i1, i2)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            preds = torch.max(outputs, 1)[1]
            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            tn += ((preds == 0) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()
            
    avg_loss = running_loss / len(loader)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = f1 # Dice is equivalent to F1 in binary classification
    
    return avg_loss, dice, iou, precision, recall, f1

# --- Main Training Loop ---
def run_training(config):
    # 1. Data Preparation
    extensions = ['*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG']
    images2 = []
    subfolders = [f.name for f in os.scandir(config["folder2_path"]) if f.is_dir()]
    
    if 'positive' in subfolders and 'negative' in subfolders:
        print("Using subfolder labels...")
        pos_imgs = []
        neg_imgs = []
        for ext in extensions:
            pos_imgs.extend(glob(os.path.join(config["folder2_path"], 'positive', ext)))
            neg_imgs.extend(glob(os.path.join(config["folder2_path"], 'negative', ext)))
        images2 = pos_imgs + neg_imgs
        labels = np.array([1] * len(pos_imgs) + [0] * len(neg_imgs))
        images1 = [os.path.join(config["folder1_path"], os.path.basename(p)) for p in images2]
    else:
        print("Loading images directly (falling back to split-labels)...")
        images1 = []
        for ext in extensions:
            images1.extend(glob(os.path.join(config["folder1_path"], ext)))
        images1 = sorted(images1)
        images2 = [os.path.join(config["folder2_path"], os.path.basename(p)) for p in images1]
        labels = np.array([0 if i < len(images1)//2 else 1 for i in range(len(images1))])

    pairs = list(zip(images1, images2))
    if len(pairs) == 0:
        print(f"Error: No images found in {config['folder1_path']}")
        return

    # 2. Training Components
    kf = KFold(n_splits=config["k_folds"], shuffle=True, random_state=42)
    fold_metrics = []
    
    transform = transforms.Compose([
        transforms.Resize((config["image_size"], config["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # 3. K-Fold Loop
    for fold, (train_idx, val_idx) in enumerate(kf.split(pairs)):
        print(f"\n--- FOLD {fold+1} ---")
        
        full_dataset = PairedImageDataset(pairs, labels, transform)
        train_sub = Subset(full_dataset, train_idx)
        val_sub = Subset(full_dataset, val_idx)
        
        train_loader = DataLoader(train_sub, batch_size=config["batch_size"], shuffle=True)
        val_loader = DataLoader(val_sub, batch_size=config["batch_size"], shuffle=False)

        model = PairedResNet(config["num_classes"]).to(device)
        optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])
        criterion = nn.CrossEntropyLoss()
        
        save_path = f"best_model_fold{fold+1}.pth"
        early_stopping = EarlyStopping(patience=config["patience"])

        for epoch in range(config["num_epochs"]):
            model.train()
            train_loss = 0
            for i1, i2, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
                i1, i2, lbls = i1.to(device), i2.to(device), lbls.to(device)
                optimizer.zero_grad()
                loss = criterion(model(i1, i2), lbls)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            v_loss, v_dice, v_iou, v_prec, v_rec, v_f1 = evaluate(model, val_loader, criterion)
            print(f"Fold {fold+1} | Epoch {epoch+1} | Val Loss: {v_loss:.4f} | Dice: {v_dice:.4f}")
            
            early_stopping(v_loss, model, save_path)
            if early_stopping.early_stop:
                print("Early stopping triggered.")
                break
        
        # Load best and evaluate
        model.load_state_dict(torch.load(save_path))
        _, f_dice, f_iou, f_prec, f_rec, f_f1 = evaluate(model, val_loader, criterion)
        fold_metrics.append([f_dice, f_iou, f_prec, f_rec, f_f1])

    # 4. Save CSV with specified sequence
    cols = ['Dice', 'IoU', 'precision', 'recall', 'F1-score']
    df = pd.DataFrame(fold_metrics, columns=cols)
    df.loc['Average'] = df.mean()
    df.to_csv('kfold_segmentation_metrics.csv', index=True)
    print("\nResults saved to kfold_segmentation_metrics.csv")

if __name__ == "__main__":
    run_training(config)

Using device: cuda
Loading images directly (falling back to split-labels)...

--- FOLD 1 ---


Epoch 1: 100%|█████████████████████████████████████████████████████████████████████████| 50/50 [00:24<00:00,  2.00it/s]


Fold 1 | Epoch 1 | Val Loss: 0.6418 | Dice: 0.6374


Epoch 2: 100%|█████████████████████████████████████████████████████████████████████████| 50/50 [00:10<00:00,  4.91it/s]


Fold 1 | Epoch 2 | Val Loss: 0.9258 | Dice: 0.6937


...

Fold 5 | Epoch 10 | Val Loss: 0.4162 | Dice: 0.8772


Epoch 11: 100%|████████████████████████████████████████████████████████████████████████| 50/50 [00:10<00:00,  4.83it/s]


Fold 5 | Epoch 11 | Val Loss: 0.4743 | Dice: 0.8729


Epoch 12: 100%|████████████████████████████████████████████████████████████████████████| 50/50 [00:10<00:00,  4.84it/s]


Fold 5 | Epoch 12 | Val Loss: 0.4155 | Dice: 0.8785
Early stopping triggered.

Results saved to kfold_segmentation_metrics.csv


In [7]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold

# 1. Redefining evaluate to include Loss and Accuracy (Fixed Assignment)
def evaluate_complete(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    # Fixed: 4 variables = 4 zeros
    tp, fp, tn, fn = 0, 0, 0, 0 
    correct = 0
    total = 0
    
    with torch.no_grad():
        for i1, i2, labels in loader:
            i1, i2, labels = i1.to(device), i2.to(device), labels.to(device)
            outputs = model(i1, i2)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            preds = torch.max(outputs, 1)[1]
            
            # Accuracy components
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            
            # Segmentation/Classification components (True Positive, False Positive, etc.)
            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            tn += ((preds == 0) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()
            
    avg_loss = running_loss / len(loader)
    accuracy = correct / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = f1 
    
    return avg_loss, dice, iou, precision, recall, f1, accuracy

# 2. Final Extraction Function
def extract_final_metrics(config, pairs, labels, transform):
    all_fold_results = []
    kf = KFold(n_splits=config["k_folds"], shuffle=True, random_state=42)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(pairs)):
        print(f"Extracting Fold {fold+1}...")
        
        full_dataset = PairedImageDataset(pairs, labels, transform)
        train_sub = Subset(full_dataset, train_idx)
        val_sub = Subset(full_dataset, val_idx)
        
        train_loader = DataLoader(train_sub, batch_size=config["batch_size"], shuffle=False)
        val_loader = DataLoader(val_sub, batch_size=config["batch_size"], shuffle=False)
        
        # Load best weights for this specific fold
        model = PairedResNet(config["num_classes"]).to(device)
        model.load_state_dict(torch.load(f"best_model_fold{fold+1}.pth"))
        criterion = nn.CrossEntropyLoss()
        
        # Calculate metrics for both sets
        t_loss, t_dice, t_iou, t_prec, t_rec, t_f1, t_acc = evaluate_complete(model, train_loader, criterion)
        v_loss, v_dice, v_iou, v_prec, v_rec, v_f1, v_acc = evaluate_complete(model, val_loader, criterion)
        
        all_fold_results.append({
            'Fold': fold + 1,
            # Training Metrics Sequence
            'Train_Loss': t_loss, 'Train_Dice': t_dice, 'Train_IoU': t_iou, 
            'Train_Precision': t_prec, 'Train_Recall': t_rec, 'Train_F1': t_f1, 'Train_Accuracy': t_acc,
            # Validation Metrics Sequence
            'Val_Loss': v_loss, 'Val_Dice': v_dice, 'Val_IoU': v_iou, 
            'Val_Precision': v_prec, 'Val_Recall': v_rec, 'Val_F1': v_f1, 'Val_Accuracy': v_acc
        })

    # Create DataFrame and enforce specific column order
    df_results = pd.DataFrame(all_fold_results)
    column_order = ['Fold', 
                    'Train_Loss', 'Train_Dice', 'Train_IoU', 'Train_Precision', 'Train_Recall', 'Train_F1', 'Train_Accuracy',
                    'Val_Loss', 'Val_Dice', 'Val_IoU', 'Val_Precision', 'Val_Recall', 'Val_F1', 'Val_Accuracy']
    df_results = df_results[column_order]
    
    # Add Mean row at the bottom
    df_results.loc['Mean'] = df_results.mean(numeric_only=True)
    df_results.at['Mean', 'Fold'] = 'AVERAGE'
    
    # Final CSV Save
    df_results.to_csv('final_kfold_train_val_metrics.csv', index=False)
    
    print("\n--- Mean Results Across All Folds ---")
    print(df_results.iloc[-1].drop('Fold'))
    return df_results

# 3. Execute Extraction
# Ensure pairs, labels, and transform were initialized in the previous step
final_df = extract_final_metrics(config, pairs, labels, transform)

Extracting Fold 1...
Extracting Fold 2...
Extracting Fold 3...
Extracting Fold 4...
Extracting Fold 5...

--- Mean Results Across All Folds ---
Train_Loss         0.234304
Train_Dice         0.908214
Train_IoU          0.835293
Train_Precision    0.897937
Train_Recall       0.926373
Train_F1           0.908214
Train_Accuracy     0.906375
Val_Loss           0.430163
Val_Dice           0.812208
Val_IoU            0.689131
Val_Precision      0.783974
Val_Recall           0.8553
Val_F1             0.812208
Val_Accuracy         0.8075
Name: Mean, dtype: object


C:\Users\imonm\AppData\Local\Temp\ipykernel_3760\3676963091.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'AVERAGE' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_results.at['Mean', 'Fold'] = 'AVERAGE'
